In [ ]:
from pyhydra.utils import interactive_map
from pyhydra.data_sources.rainfall import download_era5
import os
from pathlib import Path

# 🌐 ERA5: Reanalysis Data by Copernicus Climate Change Service (C3S)

## 📝 **What is ERA5?**

[ERA5](https://www.ecmwf.int/en/forecasts/datasets/reanalysis-datasets/era5) is the fifth-generation reanalysis dataset produced by the **European Centre for Medium-Range Weather Forecasts (ECMWF)** under the **Copernicus Climate Change Service (C3S)**.

It provides a comprehensive reconstruction of the global climate system, combining **historical observations** with advanced atmospheric models to produce **consistent, hourly estimates** of multiple variables related to the atmosphere, land, and oceans.

---

## 🗂 **Data Source and Coverage**

- **Institution:** European Centre for Medium-Range Weather Forecasts (ECMWF)  
- **Program:** Copernicus Climate Change Service (C3S)  
- **Temporal coverage:** From **1950** (back extension) or **1979** (full coverage) to near real-time (~5-day delay).  
- **Spatial resolution:** ~31 km (0.25° x 0.25°) globally.  
- **Temporal resolution:** Hourly (for single levels dataset).

---

## 📦 **Available Datasets and Temporal Resolution**

| **Dataset**                                | **Temporal resolution** | **Description**                            |
|---------------------------------------------|--------------------------|---------------------------------------------|
| **ERA5 Single Levels**                    | Hourly, Monthly Means  | Surface and single level atmospheric variables. |
| **ERA5 Pressure Levels**                  | Hourly                 | Atmospheric variables at multiple pressure levels. |
| **ERA5 Land**                             | Hourly                 | High-resolution land variables (~9 km).    |
| **ERA5 Wave**                             | Hourly                 | Ocean wave parameters.                    |

> ⚠️ **Note:**  
> The current code is **only prepared for ERA5 Single Levels**, both **hourly** and **monthly means**.  
> Downloading pressure levels, ERA5 Land, or wave data requires adapting the dataset endpoint and request parameters accordingly.

---

## 🌤 **Variables in ERA5 Single Levels**

Common variables include:

- 2m temperature
- 2m dewpoint temperature
- Total precipitation
- Surface runoff
- Sea level pressure
- Surface pressure
- 10m wind components (u10, v10)
- Shortwave and longwave radiation fluxes

You can find the **complete list of variables and their descriptions** at:

- [CDS variable names - Variable List](https://confluence.ecmwf.int/display/CKB/ERA5%3A+data+documentation#ERA5:datadocumentation-Parameterlistings)


> 💡 **Tip:** Each dataset page provides a form with checkboxes for all available variables, units, and definitions.

---

### 🔗 **Full Variable Listings**

You can find the **complete list of variables and their descriptions** at:

- [ERA5 Single Levels - Variable List](https://cds.climate.copernicus.eu/cdsapp#!/dataset/reanalysis-era5-single-levels?tab=form)
- [ERA5 Monthly Means - Variable List](https://cds.climate.copernicus.eu/cdsapp#!/dataset/reanalysis-era5-single-levels-monthly-means?tab=form)

For other datasets:

- [ERA5 Pressure Levels - Variable List](https://cds.climate.copernicus.eu/cdsapp#!/dataset/reanalysis-era5-pressure-levels?tab=form)
- [ERA5 Land - Variable List](https://cds.climate.copernicus.eu/cdsapp#!/dataset/reanalysis-era5-land?tab=form)
- [ERA5 Wave - Variable List](https://cds.climate.copernicus.eu/cdsapp#!/dataset/reanalysis-era5-single-levels?tab=form)

---

## 🔗 **Where to Access ERA5 Data**

- **Copernicus Climate Data Store (CDS) main portal:**  
  [https://cds.climate.copernicus.eu/cdsapp#!/home](https://cds.climate.copernicus.eu/cdsapp#!/home)

---

## ✅ **Key Advantages of ERA5**

- **High spatial and temporal resolution** with global coverage.  
- **Consistent, quality-controlled datasets** suitable for climate analysis, hydrological modeling, and impact assessments.  
- Availability of **ensemble members** for uncertainty quantification.

---

## 📚 **Further Reading and Documentation**

- [Copernicus ERA5 Overview](https://climate.copernicus.eu/climate-reanalysis)  
- [ERA5 Data Documentation](https://confluence.ecmwf.int/display/CKB/ERA5%3A+data+documentation)  
- [ECMWF Reanalysis Datasets](https://www.ecmwf.int/en/forecasts/datasets/reanalysis-datasets)

---

*Prepared for integration into climate data processing pipelines, reproducible research, and documentation.*


## 🗺️ Select area of interest

Draw a rectangle on the map below to define the download region. Read `coordinates_list[0]` in the next cell.

In [ ]:
coordinates_list = interactive_map(zoom=4, center=(40, -5))

In [ ]:
# === CONFIGURATION ===
# Credentials are read from the environment. Do not commit personal API keys.
url = os.getenv('CDS_API_URL', 'https://cds.climate.copernicus.eu/api')
api_key = os.getenv('CDS_API_KEY') or os.getenv('CDSAPI_KEY')

# Release/build mode executes notebooks without large external downloads.
# Set HYDRA_DOWNLOAD_SMOKE=1 for a tiny API/download check.
# Set HYDRA_RUN_DOWNLOADS=1 for the full historical download.
RUN_DOWNLOADS = os.getenv('HYDRA_RUN_DOWNLOADS', '0') == '1'
RUN_SMOKE = os.getenv('HYDRA_DOWNLOAD_SMOKE', '0') == '1'

area = coordinates_list[0] if coordinates_list else [44, -10, 35, 5]  # [N, W, S, E]
download_base_dir = os.getenv('HYDRA_ERA5_DIR', '/workspace/data/era5/')
os.makedirs(download_base_dir, exist_ok=True)

variables_list = ['total_precipitation', 'surface_runoff', '2m_temperature']
year_ini = 1990
year_end = 2020

print(f'ERA5 output dir : {download_base_dir}')
print(f'Run full download: {RUN_DOWNLOADS}')
print(f'Run smoke check  : {RUN_SMOKE}')
print(f'API key present  : {bool(api_key)}')

ERA5 output dir : /tmp/hydra_era5_smoke
Run full download: False
Run smoke check  : True
API key present  : True


In [ ]:
if RUN_DOWNLOADS or RUN_SMOKE:
    if not api_key:
        print('CDS API key not configured. Set CDS_API_KEY or CDSAPI_KEY to run downloads.')
    elif RUN_SMOKE:
        smoke_dir = str(Path(download_base_dir) / 'smoke')
        smoke_area = [43.35, -4.15, 43.15, -3.95]  # small box around Los Corrales de Buelna
        print('Running ERA5 smoke download: 1 variable, 1 month, monthly means, small area...')
        download_era5(
            folder_out=smoke_dir,
            api_key=api_key,
            url=url,
            area=smoke_area,
            variables_list=['total_precipitation'],
            years=[2020],
            months=[1],
            max_workers=1,
            file_format='netcdf',
            frequency='monthly',
        )
        print(f'ERA5 smoke output -> {smoke_dir}')
    else:
        print('Running full ERA5 download. This is intentionally disabled during image builds.')
        download_era5(
            folder_out=download_base_dir,
            api_key=api_key,
            url=url,
            area=area,
            variables_list=variables_list,
            years=range(year_ini, year_end + 1),
            months=range(1, 13),
            max_workers=5,
            file_format='netcdf',
            frequency='hourly',
        )
else:
    print('Skipping ERA5 download in release mode. Set HYDRA_DOWNLOAD_SMOKE=1 for a tiny API test or HYDRA_RUN_DOWNLOADS=1 for the full download.')

Running ERA5 smoke download: 1 variable, 1 month, monthly means, small area...
Saved: /tmp/hydra_era5_smoke/smoke/ERA5_monthly_2020_01_combined.nc (elapsed: 0:00:26.222410)
ERA5 smoke output -> /tmp/hydra_era5_smoke/smoke


2026-06-13 12:01:27,568 INFO [2026-06-11T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 15 June. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure-part-2/150414).
2026-06-13 12:01:28,229 INFO Request ID is 749885b5-47b1-4987-bd9d-747c3a617d55
2026-06-13 12:01:30,316 INFO status has been updated to accepted
2026-06-13 12:01:44,089 INFO status has been updated to running
2026-06-13 12:01:51,820 INFO status has been updated to successful
